# Evaluación RAG — Fase 3: Métricas Financieras

Cruza el sentimiento generado por el LLM con la econometría real del corpus (`return_1d/5d/20d`, `market_reaction_1d/5d/20d`). No necesita Pinecone ni embeddings — reutiliza el **mismo archivo de predicciones de la Fase 2** (`id` + `sentimiento_predicho`).

Métricas implementadas:
- **Directional Accuracy** por horizonte (1d/5d/20d)
- **Information Coefficient (IC)** — Spearman (principal) y Pearson (secundaria) entre sentimiento y retorno
- **Jonckheere-Terpstra** — test de tendencia ordinal Bearish→Sideways→Bullish
- **Spread Bullish–Bearish** con t-test de significancia
- **Time-Lag Analysis** — Directional Accuracy + IC por horizonte, desglosado por categoría de periodo

### Documentos necesarios
1. `financial_ground_truth.csv` (adjunto) — igual que el ground truth de sentimiento, pero con `return_*`, `market_reaction_*`, `price_*` y `enrich_status` añadidos.
2. **El mismo archivo de predicciones que usaste en la Fase 2** (`id` + `sentimiento_predicho`) — no hace falta generarlo de nuevo.

Se excluyen automáticamente las filas con `enrich_status != "ok"` (82 de 1391, sin datos de precio válidos).

In [ ]:
# @title 1. Instalación de dependencias
!pip install -q pandas scipy matplotlib seaborn

## 2. Subir archivos

In [ ]:
from google.colab import files
uploaded = files.upload()
print("Archivos subidos:", list(uploaded.keys()))

## 3. Configuración

In [ ]:
# @title Configuración
GROUND_TRUTH_PATH = "financial_ground_truth.csv"  # @param {type:"string"}
PREDICTIONS_PATH = "predicciones_sentimiento_sonnet.json"  # @param {type:"string"}
PREDICTIONS_FORMAT = "json"  # @param ["csv", "json"]
HORIZONTES = ["1d", "5d", "20d"]

assert PREDICTIONS_PATH, "Indica el nombre del archivo de predicciones que subiste."


## 4. Normalización de etiquetas (idéntica a la Fase 2)

In [ ]:
import re

LABEL_MAP = {
    "bearish": "bearish", "bajista": "bearish", "bajo": "bearish", "negativo": "bearish",
    "bear": "bearish", "down": "bearish", "venta": "bearish", "vender": "bearish",
    "bullish": "bullish", "alcista": "bullish", "alto": "bullish", "positivo": "bullish",
    "bull": "bullish", "up": "bullish", "compra": "bullish", "comprar": "bullish",
    "sideways": "sideways", "lateral": "sideways", "neutral": "sideways", "neutro": "sideways",
    "estable": "sideways", "flat": "sideways", "sin cambios": "sideways",
}
SCORE_MAP = {"bearish": -1, "sideways": 0, "bullish": 1}

def normalize_label(raw):
    if raw is None:
        return None
    key = str(raw).strip().lower()
    key = re.sub(r"[^a-záéíóúñ ]", "", key)
    return LABEL_MAP.get(key, None)

## 5. Cargar, unir y filtrar

In [ ]:
import pandas as pd

gt = pd.read_csv(GROUND_TRUTH_PATH)

if PREDICTIONS_FORMAT == "csv":
    preds = pd.read_csv(PREDICTIONS_PATH)
else:
    preds = pd.read_json(PREDICTIONS_PATH)

preds["sentimiento_predicho_norm"] = preds["sentimiento_predicho"].apply(normalize_label)
preds["score_sentimiento"] = preds["sentimiento_predicho_norm"].map(SCORE_MAP)

df = gt.merge(preds[["id", "sentimiento_predicho_norm", "score_sentimiento"]], on="id", how="inner")
df = df.dropna(subset=["sentimiento_predicho_norm"])

n_antes = len(df)
df = df[df["enrich_status"] == "ok"].copy()
print(f"Filas tras unión con predicciones: {n_antes}")
print(f"Filas tras excluir enrich_status != 'ok': {len(df)}  (excluidas: {n_antes - len(df)})")

df.head()

## 6. Directional Accuracy por horizonte

`Directional Accuracy@horizonte = % noticias donde sentimiento_RAG == market_reaction_{horizonte}`

In [ ]:
directional_accuracy = {}
for h in HORIZONTES:
    col = f"market_reaction_{h}"
    valid = df[df[col].notna()]
    acc = (valid["sentimiento_predicho_norm"] == valid[col]).mean()
    directional_accuracy[h] = acc
    print(f"Directional Accuracy @{h}: {acc:.4f}  (n={len(valid)})")

## 7. Information Coefficient (IC) — Spearman (principal) y Pearson (secundaria)

In [ ]:
from scipy.stats import spearmanr, pearsonr

ic_results = []
for h in HORIZONTES:
    col = f"return_{h}"
    valid = df[df[col].notna()]
    ic_spearman, p_spearman = spearmanr(valid["score_sentimiento"], valid[col])
    ic_pearson, p_pearson = pearsonr(valid["score_sentimiento"], valid[col])
    ic_results.append({
        "horizonte": h, "n": len(valid),
        "IC_spearman": ic_spearman, "p_valor_spearman": p_spearman,
        "IC_pearson": ic_pearson, "p_valor_pearson": p_pearson,
    })

ic_df = pd.DataFrame(ic_results)
ic_df

## 8. Jonckheere-Terpstra — test de tendencia ordinal

Contrasta si existe una tendencia monótona real Bearish < Sideways < Bullish en el retorno. No viene incluido en scipy, así que se implementa aquí: para cada par de grupos ordenados, se suma el estadístico U de Mann-Whitney, y se normaliza a un z-score bajo la aproximación normal estándar del test.

In [ ]:
import numpy as np
from scipy.stats import norm, mannwhitneyu

def jonckheere_terpstra(groups):
    """groups: lista de arrays de valores, en orden ordinal ascendente (ej. [bearish, sideways, bullish]).
    Devuelve (J estadístico, z, p-valor de una cola: tendencia ascendente)."""
    k = len(groups)
    n = [len(g) for g in groups]
    N = sum(n)

    J = 0
    for i in range(k):
        for j in range(i + 1, k):
            # U_ij = número de pares (x en groups[i], y en groups[j]) con y > x
            u_stat, _ = mannwhitneyu(groups[i], groups[j], alternative="less")
            # mannwhitneyu con alternative='less' testea groups[i] < groups[j];
            # el estadístico U devuelto por scipy corresponde a U(groups[i], groups[j])
            J += n[i] * n[j] - u_stat  # convertir a "número de pares con groups[j] > groups[i]"

    mean_J = (N**2 - sum(ni**2 for ni in n)) / 4
    var_J = (N**2 * (2 * N + 3) - sum(ni**2 * (2 * ni + 3) for ni in n)) / 72
    z = (J - mean_J) / np.sqrt(var_J)
    p_value = 1 - norm.cdf(z)  # una cola: tendencia ascendente Bearish->Bullish
    return J, z, p_value


jt_results = []
for h in HORIZONTES:
    col = f"return_{h}"
    valid = df[df[col].notna()]
    groups = [
        valid[valid["sentimiento_predicho_norm"] == "bearish"][col].values,
        valid[valid["sentimiento_predicho_norm"] == "sideways"][col].values,
        valid[valid["sentimiento_predicho_norm"] == "bullish"][col].values,
    ]
    J, z, p = jonckheere_terpstra(groups)
    jt_results.append({"horizonte": h, "J": J, "z": z, "p_valor": p, "tendencia_significativa (p<0.05)": p < 0.05})

jt_df = pd.DataFrame(jt_results)
jt_df

## 9. Spread Bullish–Bearish (señal tipo long-short) + t-test

In [ ]:
from scipy.stats import ttest_ind

spread_results = []
for h in HORIZONTES:
    col = f"return_{h}"
    valid = df[df[col].notna()]
    bullish_returns = valid[valid["sentimiento_predicho_norm"] == "bullish"][col]
    bearish_returns = valid[valid["sentimiento_predicho_norm"] == "bearish"][col]

    spread = bullish_returns.mean() - bearish_returns.mean()
    t_stat, p_value = ttest_ind(bullish_returns, bearish_returns, equal_var=False, nan_policy="omit")

    spread_results.append({
        "horizonte": h,
        "n_bullish": len(bullish_returns), "n_bearish": len(bearish_returns),
        "mean_return_bullish": bullish_returns.mean(), "mean_return_bearish": bearish_returns.mean(),
        "spread": spread, "t_stat": t_stat, "p_valor": p_value,
        "significativo (p<0.05)": p_value < 0.05,
    })

spread_df = pd.DataFrame(spread_results)
spread_df

## 10. Time-Lag Analysis

Directional Accuracy e IC (Spearman) por horizonte, desglosados por categoría de periodo (Caídas / Subidas / Lateralización), para identificar en qué horizonte temporal el sentimiento del RAG es más predictivo según el tipo de narrativa.

In [ ]:
def categoria_periodo(periodo):
    p = periodo.lower()
    if p.startswith("lateral"):
        return "Lateralización"
    if p.startswith("caidas"):
        return "Caídas"
    if p.startswith("subidas"):
        return "Subidas"
    return "otro"

df["categoria_periodo"] = df["periodo"].apply(categoria_periodo)

lag_rows = []
for categoria, sub in df.groupby("categoria_periodo"):
    for h in HORIZONTES:
        col_return = f"return_{h}"
        col_reaction = f"market_reaction_{h}"
        valid = sub[sub[col_return].notna()]
        if len(valid) < 5:
            continue
        acc = (valid["sentimiento_predicho_norm"] == valid[col_reaction]).mean()
        ic, p_val = spearmanr(valid["score_sentimiento"], valid[col_return])
        lag_rows.append({
            "categoria": categoria, "horizonte": h, "n": len(valid),
            "directional_accuracy": acc, "IC_spearman": ic, "p_valor": p_val,
        })

lag_df = pd.DataFrame(lag_rows)
pivot_acc = lag_df.pivot(index="categoria", columns="horizonte", values="directional_accuracy")
pivot_ic = lag_df.pivot(index="categoria", columns="horizonte", values="IC_spearman")

print("Directional Accuracy por categoría y horizonte:")
display(pivot_acc[HORIZONTES].round(4))
print("\nInformation Coefficient (Spearman) por categoría y horizonte:")
display(pivot_ic[HORIZONTES].round(4))

## 11. Visualización — IC por horizonte

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 5))
ax.bar(ic_df["horizonte"], ic_df["IC_spearman"], color=["#B5474D", "#C7A24D", "#3E8E58"])
ax.axhline(0, color="black", linewidth=0.8)
ax.set_ylabel("Information Coefficient (Spearman)")
ax.set_title("IC por horizonte temporal")
for i, row in ic_df.iterrows():
    marker = "*" if row["p_valor_spearman"] < 0.05 else ""
    ax.text(i, row["IC_spearman"], f"{row['IC_spearman']:.3f}{marker}", ha="center",
            va="bottom" if row["IC_spearman"] >= 0 else "top")
plt.tight_layout()
plt.savefig("ic_por_horizonte.png", dpi=150)
plt.show()
print("* = estadísticamente significativo (p<0.05)")

## 12. Guardar resultados

In [ ]:
directional_df = pd.DataFrame([{"horizonte": h, "directional_accuracy": v} for h, v in directional_accuracy.items()])

with pd.ExcelWriter("resultados_financieros.xlsx") as writer:
    directional_df.to_excel(writer, sheet_name="directional_accuracy", index=False)
    ic_df.to_excel(writer, sheet_name="information_coefficient", index=False)
    jt_df.to_excel(writer, sheet_name="jonckheere_terpstra", index=False)
    spread_df.to_excel(writer, sheet_name="spread_bullish_bearish", index=False)
    lag_df.to_excel(writer, sheet_name="time_lag_analysis", index=False)

from google.colab import files as colab_files
colab_files.download("resultados_financieros.xlsx")
colab_files.download("ic_por_horizonte.png")

print("Guardado resultados_financieros.xlsx con 5 hojas.")